# NodeSubstrates - Insurance Fraud Network Demo

This notebook demonstrates NodeSubstrates on an **insurance fraud detection network**.
The dataset contains claims data with relationships between accidents, cars, lawyers, 
doctors, and participants (drivers, passengers, witnesses).

**Use Case**: Insurance investigators use network analysis to detect organized fraud rings 
where the same lawyers, doctors, or participants appear across multiple suspicious claims.

In [5]:
import importlib
import node_substrates
importlib.reload(node_substrates)

from node_substrates import NodeSubstratesWidget
from node_substrates.datasets.insurancefraud import load_insurance_fraud_network

NodeSubstratesWidget._esm._contents = None
NodeSubstratesWidget._css._contents = None

## 1. Load the Insurance Fraud Network

The dataset contains nodes representing:
- **Accident**: Insurance claims (central events)
- **Car**: Vehicles involved in accidents
- **Lawyer**: Legal representatives
- **Doctor**: Medical professionals treating participants
- **Participant**: People involved (drivers, passengers, witnesses)

**Pre-computed attributes** include:
- Network metrics: `degree`, `clustering`, `betweenness`
- **Node2Vec embeddings**: 32-dimensional embeddings (`n2v_0` to `n2v_31`) for structural similarity

In [6]:
# Load the insurance fraud network
G = load_insurance_fraud_network()

# Select only 100 nodes for debugging/DEVELOPMENT
# nodes_subset = list(G.nodes())[:100]
# G    = G.subgraph(nodes_subset).copy()

print(f"\nNetwork: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Count by type
type_counts = {}
for node, attrs in G.nodes(data=True):
    t = attrs.get('type', 'Unknown')
    type_counts[t] = type_counts.get(t, 0) + 1

print(f"\nNode types:")
for t, count in sorted(type_counts.items()):
    print(f"  {t}: {count}")

# Sample node attributes
print(f"\nSample node attributes:")
sample_node = list(G.nodes())[0]
for key, value in G.nodes[sample_node].items():
    print(f"  {key}: {value}")

Loaded insurance fraud network: 1547 nodes, 1947 edges
Node types: {'Accident': 61, 'Car': 192, 'Lawyer': 342, 'Participant': 827, 'Doctor': 125}

Network: 1547 nodes, 1947 edges

Node types:
  Accident: 61
  Car: 192
  Doctor: 125
  Lawyer: 342
  Participant: 827

Sample node attributes:
  type: Accident
  enter: 2021-11-01
  exit: 2021-12-06
  embedding: [0.29685765504837036, -1.1920983791351318, 0.18120312690734863, 0.3863914906978607, 0.5140900611877441, 0.23203164339065552, -0.09602648764848709, 0.0055848960764706135, 1.4771548509597778, 0.1307734102010727, -0.005040179938077927, -0.7676149606704712, -0.5377259254455566, -0.38645464181900024, -1.1544712781906128, 1.0386502742767334, 0.010357867926359177, -1.3268084526062012, 1.0529687404632568, 0.9637145400047302, 1.1713037490844727, -0.4339408874511719, -0.4526655673980713, 0.1102011576294899, -1.2935779094696045, 0.6654775738716125, -1.4892566204071045, 0.18179529905319214, -1.1205652952194214, -0.07441689074039459, 1.1275070905

## 2. Create NodeSubstrates Widget

The initial view shows a force-directed layout revealing the network structure.
Fraud rings often appear as dense clusters connected by key individuals.

In [8]:
widget = NodeSubstratesWidget(G, auto_substrate=False, width=1200, height=600, initial_scale=0.40)
widget

## 3. Explore Auto-detected Substrate Suggestions

NodeSubstrates identifies regions where attribute visualization would help:
- **High-degree professionals**: Lawyers/doctors with many clients
- **Complex accidents**: Claims with many participants
- **Cross-claim connections**: Participants appearing in multiple accidents

In [ ]:
print("Auto-detected substrate suggestions:\n")
for i, suggestion in enumerate(widget.suggested_regions[:5]):  # Show top 5
    print(f"Suggestion {i}: {suggestion['label']}")
    print(f"  Nodes: {len(suggestion['node_ids'])}")
    print(f"  Score: {suggestion['score']:.3f}")
    print(f"  Reason: {suggestion['reason']}")
    print(f"  Recommended DR: {suggestion['recommended_dr']}")
    print()

## 4. Create a Substrate for High-Connectivity Nodes

Let's create a substrate focusing on the most connected professionals 
(lawyers and doctors) to see if they cluster by behavior patterns.

In [ ]:
# Select lawyers and doctors with high degree
# Note: widget uses string IDs internally
professionals = [
    str(node) for node, attrs in G.nodes(data=True) 
    if attrs.get('type') in ('Lawyer', 'Doctor') and attrs.get('degree', 0) >= 3
]

print(f"Found {len(professionals)} high-connectivity professionals")

if len(professionals) >= 3:
    substrate_id = widget.create_substrate(
        professionals,
        dr_method='umap',
        label='High-Connectivity Professionals'
    )
    print(f"Created substrate: {substrate_id}")

## 5. Compare DR Methods

Different dimensionality reduction methods reveal different patterns:
- **PCA**: Linear relationships (e.g., degree vs betweenness)
- **UMAP**: Non-linear clusters (e.g., fraud ring groupings)
- **t-SNE**: Local structure emphasis

In [ ]:
# Try UMAP for cluster discovery
if widget.substrates:
    widget.update_dr_method(widget.substrates[0]['id'], 'umap')
    print("Switched to UMAP - look for clusters of similar behavior patterns")

## 6. Interactive Fraud Investigation

Use lasso selection to investigate suspicious clusters:
- **Shift+Drag** to select nodes
- Create substrates from selections
- Compare different accident clusters side-by-side

In [ ]:
# Check current selection
print(f"Selected nodes: {widget.selected_nodes}")

# Create substrate from selection (need at least 3 nodes)
if len(widget.selected_nodes) >= 3:
    substrate_id = widget.create_substrate(
        widget.selected_nodes,
        dr_method='umap',
        label='Investigation Selection'
    )
    print(f"Created substrate from selection: {substrate_id}")

## 7. Current Substrates

In [ ]:
# List current substrates
print("Current substrates:")
for s in widget.substrates:
    print(f"  {s['id']}: {s['label']} ({len(s['node_ids'])} nodes, {s['dr_method']})")

In [ ]:
# Dissolve a substrate to return nodes to force-directed layout
if widget.substrates:
    substrate_to_dissolve = widget.substrates[0]['id']
    widget.dissolve_substrate(substrate_to_dissolve)
    print(f"Dissolved {substrate_to_dissolve}")

## 8. Fraud Detection Insights

The hybrid NodeSubstrates view helps investigators by:

**Network View (Force-Directed)**:
- Reveals accident clusters and their connections
- Shows which professionals span multiple claims
- Identifies isolated vs. connected incidents

**Attribute View (Substrates)**:
- Groups similar professionals by behavior metrics
- Highlights outliers with unusual patterns
- Enables comparison of node characteristics

**Key Fraud Indicators**:
- Lawyers/doctors appearing in many unrelated claims
- Participants involved in multiple accidents
- Clusters with unusually high connectivity
- Bridge nodes connecting otherwise separate claim groups

In [ ]:
# Summary statistics
print(f"\n=== Insurance Fraud Network Summary ===")
print(f"Total nodes: {len(widget.nodes)}")
print(f"Nodes in substrates: {len(widget.substrate_node_ids)}")
print(f"Nodes in force-directed: {len(widget.topological_node_ids)}")
print(f"Active substrates: {len(widget.substrates)}")

# Type breakdown
print(f"\n=== Node Type Distribution ===")
for t, count in sorted(type_counts.items()):
    print(f"  {t}: {count}")